# GeoSR-4 — Synthetic-degradation pretraining (Kaggle GPU)

Kaggle version of `train_swinir_synthetic_pretrain_colab.ipynb` (D048). Tests whether pretraining on a
synthetic NAIP-degradation corpus (opensr-degradation, real HR-only NAIP tiles turned into synthetic
Sentinel-2-like LR) helps before fine-tuning on the real SEN2NAIP pairs, vs training on real data alone.

**Fair comparison target: D040's 7a (ICNR-only, --amp, 20 real epochs, PSNR 16.54 dB)** -- same ICNR default,
same `--amp`, same 20-epoch finetune count, batch-size 16, embed_dim 60.

**Data scale**: ~500 locations attempted (20 curated + 480 random CONUS points), expect ~300-320 successful
pairs (some land in ocean/no-coverage areas and are skipped) -- a verified run got 316/500. Real SEN2NAIP train
set is 2283 pairs, so this synthetic corpus is still smaller (proof-of-concept scale, not yet "abundant
synthetic data" scale) -- see decisions.md D048 for the honest caveat.

**Before running, in the notebook's right-hand Settings panel:**
- Accelerator: **GPU T4 x2** or **P100** -- `--amp` is already wired in for T4.
- Internet: ON (needed for git clone, dataset + NAIP tile downloads, and the opensr-degradation package).
  If you hit `Could not resolve host`, your Kaggle account likely isn't phone-verified yet
  (Settings → Account → Phone Verification), or the internet toggle needs a session restart to take effect.

**Timing**: corpus generation (~2 hours, CPU/network-bound) + pretrain+finetune GPU training (~1-1.5 hours) --
budget ~3-3.5 hours total. Kaggle's ~30 GPU-hours/week and 12-hour session cap both comfortably cover this in
one sitting, unlike Colab's free-tier usage limits which have been hit repeatedly this project (D041/D042).

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies
Kaggle's default image already has PyTorch with CUDA -- only the packages it's missing get installed.

In [ ]:
!git clone https://github.com/Vijay6923/GeoSR-4.git
%cd GeoSR-4
!pip install -q rasterio huggingface_hub scikit-image torchvision opensr-degradation einops datasets pystac-client planetary-computer

## 2. Download the real SEN2NAIP dataset (needed for fine-tuning + validation)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os

zip_path = hf_hub_download(
    repo_id="isp-uv-es/SEN2NAIP",
    repo_type="dataset",
    filename="cross-sensor/cross-sensor.zip",
    local_dir="ml/datasets/raw/sen2naip",
)

extract_dir = "ml/datasets/raw/sen2naip/cross-sensor/extracted"
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)

print("extracted to", extract_dir)

## 3. Generate the synthetic pretraining corpus
Fetches real HR-only NAIP tiles from ~500 diverse US locations (free, no auth, via Microsoft Planetary Computer)
and degrades each into a synthetic (LR, HR) pair. Some random points land in ocean/no-coverage areas and are
skipped -- expect roughly 300-320 successful pairs, not exactly 500. Takes ~2 hours (network + CPU-bound, no
GPU used yet).

In [ ]:
!python ml/datasets/generate_synthetic_pairs.py --n 20 --random-n 480 --seed 42 --start-index 20

## 4. Pretrain on synthetic corpus, then fine-tune on real data
Phase 1 (pretrain) evaluates on the *real* val split throughout, so you can watch it converge toward real-data
performance even while only training on synthetic pairs. Phase 2 (fine-tune) continues the same model on the
real train split -- same protocol as D040's 7a (20 epochs, batch 16, `--amp`) for a fair comparison. Checkpoints
save every epoch to `experiments/swinir_synthetic_pretrain/` -- if this session gets interrupted, **Save
Version** first (top right) so `/kaggle/working/` contents are preserved.

In [ ]:
!python ml/training/train_swinir_synthetic_pretrain.py \
  --pretrain-epochs 40 \
  --finetune-epochs 20 \
  --batch-size 4 \
  --finetune-batch-size 16 \
  --embed-dim 60 \
  --depths 2,2,2,2 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/swinir_synthetic_pretrain \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_synthetic_pretrain/swinir_finetune_epoch19.pt \
  --model-type swinir --embed-dim 60 --depths 2,2,2,2 --num-heads 6 --window-size 11

## 5. Optional: bigger model + longer training (run instead of section 4, not in addition)
Combines the other queued backlog item ("bigger/longer retrain") with the same synthetic-pretrain pipeline --
`embed_dim` 60→120 and RSTB depths 4 blocks→6 blocks each, pretrain-epochs 40→60, finetune-epochs 20→40. Expect
several hours on a T4 -- watch the first few epochs' loss for NaN/divergence before walking away.

In [ ]:
!python ml/training/train_swinir_synthetic_pretrain.py \
  --pretrain-epochs 60 \
  --finetune-epochs 40 \
  --batch-size 4 \
  --finetune-batch-size 16 \
  --embed-dim 120 \
  --depths 6,6,6,6 \
  --num-heads 6 \
  --window-size 11 \
  --lr 1e-4 \
  --amp \
  --checkpoint-dir experiments/swinir_bigger_synthetic_pretrain \
  --log-every 20

!python ml/evaluation/evaluate_checkpoint.py --checkpoint experiments/swinir_bigger_synthetic_pretrain/swinir_finetune_epoch39.pt \
  --model-type swinir --embed-dim 120 --depths 6,6,6,6 --num-heads 6 --window-size 11

## 6. Get the checkpoint out
Copies it to `/kaggle/working/`, downloadable from the notebook's **Output** tab once you save a version.

In [ ]:
import shutil
shutil.copy("experiments/swinir_synthetic_pretrain/swinir_finetune_epoch19.pt", "/kaggle/working/swinir_synthetic_pretrain_epoch19.pt")
print("copied -- visible in the Output tab once you save a version of this notebook")